# 00 First Steps: Python, APIs, and a Tiny Tool
### เครื่องคำนวณเวลาให้ยา + เรียก API จริงครั้งแรก

Notebook นี้คือจุดเริ่มต้นก่อน 01-clinical-ml เหมาะสำหรับคนที่ไม่เคยเขียนโค้ดมาก่อน เราจะสร้างเครื่องมือเล็ก ๆ ที่แก้ปัญหาจริง (คำนวณเวลาให้ยาครั้งถัดไป) แล้วเรียก API สาธารณะจริงครั้งแรก

> 🎯 **เป้าหมาย:** เขียนฟังก์ชัน Python ตัวแรก, เรียก API จริง, และอ่านผลลัพธ์เป็นตาราง

## 1. ตัวแปรและฟังก์ชันตัวแรก

ทุกอย่างใน Python เริ่มจากตัวแปร (เก็บค่า) และฟังก์ชัน (ทำงานซ้ำได้โดยไม่ต้องเขียนใหม่)

In [1]:
drug_name = "Paracetamol"
dose_interval_hours = 6
doses_per_day = 24 // dose_interval_hours

print(f"{drug_name}: ให้ทุก {dose_interval_hours} ชั่วโมง = {doses_per_day} ครั้งต่อวัน")

Paracetamol: ให้ทุก 6 ชั่วโมง = 4 ครั้งต่อวัน


## 2. เครื่องคำนวณเวลาให้ยาครั้งถัดไป

นี่คือ "ลองเดี๋ยวนี้" จาก [Basics](../curriculum/basics.html): เครื่องมือหน้าเดียวที่รับยา ระยะห่างโดส และเวลาโดสแรก แล้วพิมพ์โดสถัดไปสามครั้ง เขียนเป็นฟังก์ชันเดียว ใช้ซ้ำได้กับยาไหนก็ได้

In [2]:
from datetime import datetime, timedelta

def next_doses(first_dose_time: datetime, interval_hours: int, count: int = 3):
    """คืนรายการเวลาให้ยาครั้งถัดไป นับจากโดสแรก"""
    return [first_dose_time + timedelta(hours=interval_hours * i) for i in range(1, count + 1)]

first_dose = datetime(2026, 7, 2, 8, 0)
upcoming = next_doses(first_dose, dose_interval_hours)

for i, t in enumerate(upcoming, start=1):
    print(f"โดสที่ {i+1}: {t.strftime('%A %H:%M')}")

โดสที่ 2: Thursday 14:00
โดสที่ 3: Thursday 20:00
โดสที่ 4: Friday 02:00


```{tip}
ลองเปลี่ยน `drug_name`, `dose_interval_hours`, หรือ `first_dose` ข้างบนแล้วรันใหม่ นี่คือ pattern เดียวกับที่คุณจะใช้สร้างเครื่องมือจริงตลอดทั้งอคาเดมี: เขียนฟังก์ชันเล็ก ๆ ที่ทำสิ่งเดียวให้ถูกต้อง
```

## 3. เรียก API จริงครั้งแรก

API คือวิธีให้โปรแกรมหนึ่งขอข้อมูลจากอีกโปรแกรมหนึ่ง เราจะเรียก [openFDA](https://open.fda.gov/) ซึ่งเป็น API สาธารณะของ FDA สหรัฐฯ ไม่ต้องขอ API key เพื่อค้นข้อมูลฉลากยา ปลอดภัย ไม่มีข้อมูลผู้ป่วยเกี่ยวข้อง

In [3]:
import requests

resp = requests.get(
    "https://api.fda.gov/drug/label.json",
    params={"search": "openfda.brand_name:aspirin", "limit": 3},
    timeout=15,
)
print("สถานะ:", resp.status_code)
data = resp.json()
print("จำนวนผลลัพธ์ในหน้านี้:", len(data["results"]))

สถานะ: 200
จำนวนผลลัพธ์ในหน้านี้: 3


สถานะ `200` แปลว่าสำเร็จ ลองดูโครงสร้าง JSON ที่ได้กลับมา มันซ้อนกันหลายชั้น นี่คือรูปแบบที่ API ทางการแพทย์เกือบทุกตัวใช้ รวมถึง FHIR ที่คุณจะเจอใน Digital Health

In [4]:
first = data["results"][0]
print("ชื่อยา:", first.get("openfda", {}).get("brand_name"))
print("ประเภท:", first.get("openfda", {}).get("product_type"))
print("คำเตือน (ย่อ):", (first.get("warnings", ["-"])[0])[:180], "...")

ชื่อยา: ['Low Dose Aspirin']
ประเภท: ['HUMAN OTC DRUG']
คำเตือน (ย่อ): Warnings Reye's syndrome : Children and teenagers who have or are recovering from chicken pox or flu-like symptoms should not use this product. When using this product, if changes  ...


## 4. แปลงผล API เป็นตาราง

โปรแกรมจริงแทบไม่เคยอ่าน JSON ดิบ เราแปลงเป็นตารางด้วย pandas เพื่อดูและวิเคราะห์ได้ง่ายขึ้น

In [5]:
import pandas as pd

rows = []
for r in data["results"]:
    info = r.get("openfda", {})
    rows.append({
        "brand_name": (info.get("brand_name") or ["-"])[0],
        "manufacturer": (info.get("manufacturer_name") or ["-"])[0],
        "route": (info.get("route") or ["-"])[0],
    })

df = pd.DataFrame(rows)
df

,brand_name,manufacturer,route
0,Low Dose Aspirin,"P & L Development, LLC",ORAL
1,Rapidol Aspirin,Pharmadel LLC,ORAL
2,Aspirin Low Dose,ATLANTIC BIOLOGICALS CORP.,ORAL


## 5. Git คืออะไร (ไม่ต้องรัน แค่ดูรูปแบบ)

Git คือสมุดบันทึกที่จำทุกเวอร์ชันของงานคุณ คำสั่งด้านล่างไม่ใช่โค้ด Python เป็นคำสั่ง terminal ที่คุณจะใช้เมื่อ commit เครื่องคำนวณเวลาให้ยาข้างบนขึ้น GitHub จริง:

```bash
git init
git add .
git commit -m "Add medication timing calculator"
git remote add origin https://github.com/your-username/my-first-tool.git
git push -u origin main
```

อ่านเพิ่มเรื่อง Git และ GitHub ใน [Basics](../curriculum/basics.html) session ที่ 2

## สรุปและก้าวต่อไป

- เขียนฟังก์ชัน Python ตัวแรก และใช้แก้ปัญหาจริงหนึ่งอย่าง (เวลาให้ยา)
- เรียก API สาธารณะจริง อ่านสถานะ และแปลง JSON เป็นตาราง
- เห็นรูปแบบคำสั่ง Git ที่จะใช้บันทึกงานทุกชิ้นจากนี้ไป

**ลองต่อ:** เปลี่ยนคำค้นใน `params` เป็นชื่อยาอื่น เช่น `ibuprofen` แล้วดูว่าผลลัพธ์เปลี่ยนไปอย่างไร จากนั้นไปต่อที่ [01-clinical-ml](01-clinical-ml.html) เพื่อสร้างโมเดล ML ตัวแรก